In [ ]:
from IPython.display import HTML

# ------------------------------------------------------------------------------
# OPTIONAL: PASTE YOUR GOOGLE APPS SCRIPT WEB APP URL BELOW TO LOG SUBMISSIONS
# ------------------------------------------------------------------------------
GOOGLE_WEBAPP_URL = "https://script.google.com/macros/s/AKfycbx47PyTCxsgV7WAs6ccD85MA9salDbG8_vboFZnluDn-CrK5pvOxIlfdjo1alLsrV7F/exec"

colab_nse_app_code = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Past Portfolio Performance - NSE Edition</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        :root {{
            --im-red: #C8102E;
            --im-dark-red: #7A0010;
            --im-gold: #FFD700;
            --im-yellow: #FFA700;
            --im-bg: #121212;
            --im-card: #1E1E24;
            --im-text-dark: #000000;
            --im-text-light: #FFFFFF;
        }}

        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: var(--im-bg);
            color: var(--im-text-light);
            margin: 0;
            padding: 20px;
        }}

        .container {{ max-width: 1300px; margin: 0 auto; }}

        .header-card {{
            background: linear-gradient(135deg, var(--im-gold), var(--im-yellow));
            border-radius: 12px;
            padding: 18px;
            text-align: center;
            margin-bottom: 24px;
            border: 3px solid var(--im-red);
            box-shadow: 0 0 20px rgba(255, 215, 0, 0.4);
        }}

        .header-card h1 {{
            color: var(--im-text-dark);
            margin: 0 0 4px 0;
            font-size: 2.2rem;
            font-weight: 900;
            text-transform: uppercase;
            letter-spacing: 1px;
        }}

        .header-card p {{
            color: #222222;
            margin: 0;
            font-size: 0.95rem;
            font-weight: 700;
        }}

        .main-grid {{
            display: grid;
            grid-template-columns: 380px 1fr;
            gap: 20px;
        }}

        @media (max-width: 950px) {{ .main-grid {{ grid-template-columns: 1fr; }} }}

        .card {{
            background: var(--im-card);
            border-radius: 12px;
            padding: 20px;
            border: 2px solid var(--im-red);
            box-shadow: 0 4px 15px rgba(200, 16, 46, 0.25);
        }}

        label {{
            display: block;
            font-size: 0.85rem;
            font-weight: 700;
            margin-bottom: 6px;
            color: var(--im-gold);
            text-transform: uppercase;
        }}

        input, select {{
            width: 100%;
            padding: 10px;
            border: 1px solid var(--im-gold);
            background: #2A2A32;
            color: #FFFFFF;
            border-radius: 6px;
            box-sizing: border-box;
            font-size: 0.9rem;
            margin-bottom: 12px;
        }}

        .stock-row {{
            display: flex;
            gap: 8px;
            margin-bottom: 10px;
            align-items: center;
        }}

        .stock-row select {{ flex: 2.5; margin-bottom: 0; }}
        .stock-row input {{ flex: 1; margin-bottom: 0; }}

        .btn-remove {{
            background: var(--im-red);
            color: white;
            border: none;
            border-radius: 6px;
            padding: 10px 12px;
            cursor: pointer;
            font-weight: bold;
        }}

        .btn-add {{
            background: transparent;
            color: var(--im-gold);
            border: 2px dashed var(--im-gold);
            width: 100%;
            padding: 10px;
            border-radius: 6px;
            cursor: pointer;
            font-weight: 700;
            margin-top: 8px;
        }}

        .btn-calc {{
            background: linear-gradient(135deg, var(--im-red), var(--im-dark-red));
            color: var(--im-gold);
            border: 1px solid var(--im-gold);
            width: 100%;
            padding: 14px;
            border-radius: 6px;
            font-weight: 800;
            cursor: pointer;
            margin-top: 16px;
            font-size: 1.05rem;
            text-transform: uppercase;
            box-shadow: 0 0 10px rgba(200, 16, 46, 0.5);
        }}

        .btn-calc:hover {{ background: var(--im-red); color: #FFFFFF; }}

        .metrics-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
            gap: 12px;
            margin-bottom: 20px;
        }}

        .metric-card {{
            background: #262630;
            padding: 14px;
            border-radius: 8px;
            text-align: center;
            border-left: 4px solid var(--im-gold);
        }}

        .metric-title {{ font-size: 0.75rem; color: var(--im-gold); text-transform: uppercase; font-weight: 700; }}
        .metric-value {{ font-size: 1.3rem; font-weight: 800; margin-top: 4px; color: #FFFFFF; }}

        .chart-box-large {{ position: relative; height: 420px; }}

        .side-grid {{
            display: grid;
            grid-template-columns: 1fr 1.2fr;
            gap: 16px;
            margin-top: 20px;
        }}

        @media (max-width: 1000px) {{ .side-grid {{ grid-template-columns: 1fr; }} }}

        .chart-box-pie {{ position: relative; height: 300px; }}

        .table-container {{ overflow-x: auto; }}

        table {{ width: 100%; border-collapse: collapse; font-size: 0.85rem; text-align: left; }}
        th {{ background-color: var(--im-red); color: var(--im-gold); padding: 10px; border: 1px solid var(--im-dark-red); text-transform: uppercase; font-size: 0.75rem; }}
        td {{ padding: 10px; border-bottom: 1px solid #333340; color: #E0E0E0; }}
        tr:nth-child(even) {{ background-color: #24242E; }}
    </style>
</head>
<body>

<div class="container">
    <div class="header-card">
        <h1>Past Portfolio Performance</h1>
        <p>STARK INDUSTRIAL PORTFOLIO ANALYZER</p>
    </div>

    <div class="main-grid">
        <!-- INPUT CONTROLS -->
        <div class="card">
            <h3 style="color:var(--im-gold); margin-top:0;">Build Portfolio</h3>

            <div class="form-group">
                <label>Years to Backtest</label>
                <input type="number" id="timeHorizonInput" value="3" min="1" max="15" placeholder="Enter years (e.g., 1, 3, 5)">
            </div>

            <label>Select NSE Stocks & Quantities (Max 10)</label>
            <div id="stocksList"></div>

            <button class="btn-add" id="btnAddStock">+ Add Stock Slot</button>
            <button class="btn-calc" id="btnCalculate">Analyze Performance</button>
        </div>

        <!-- DASHBOARD RESULTS -->
        <div>
            <div class="metrics-grid">
                <div class="metric-card"><div class="metric-title">Current Value</div><div class="metric-value" id="valCurrent">₹0</div></div>
                <div class="metric-card"><div class="metric-title">Total Return</div><div class="metric-value" id="valReturn">0.0%</div></div>
                <div class="metric-card"><div class="metric-title">CAGR</div><div class="metric-value" id="valCAGR">0.0%</div></div>
                <div class="metric-card"><div class="metric-title">Top Performer</div><div class="metric-value" id="valBest" style="font-size:0.95rem;">-</div></div>
            </div>

            <!-- TALL LINE CHART -->
            <div class="card">
                <h4 style="color:var(--im-gold); margin:0 0 12px 0;">Growth Timeline (Base = 100 Index)</h4>
                <div class="chart-box-large">
                    <canvas id="lineChart"></canvas>
                </div>
            </div>

            <!-- SIDE BY SIDE: PIE CHART + PRICE TABLE -->
            <div class="side-grid">
                <div class="card">
                    <h4 style="color:var(--im-gold); margin:0 0 12px 0;">Portfolio Weight (%)</h4>
                    <div class="chart-box-pie">
                        <canvas id="pieChart"></canvas>
                    </div>
                </div>

                <div class="card">
                    <h4 style="color:var(--im-gold); margin:0 0 12px 0;">Holdings & Live Price Table</h4>
                    <div class="table-container">
                        <table>
                            <thead>
                                <tr>
                                    <th>Stock</th>
                                    <th>Qty</th>
                                    <th>Start Price</th>
                                    <th>Traded Price</th>
                                    <th>Value</th>
                                    <th>Weight</th>
                                </tr>
                            </thead>
                            <tbody id="holdingsTableBody"></tbody>
                        </table>
                    </div>
                </div>
            </div>

        </div>
    </div>
</div>

<script>
const LOGGING_WEBAPP_URL = "{GOOGLE_WEBAPP_URL}";

// MASTER LIST OF NSE LISTED STOCKS (YAHOO FINANCE TICKER FORMAT .NS)
const STOCK_DB = [
    {{ name: "IDFC FIRST Bank", ticker: "IDFCFIRSTB.NS", price: 82, annualReturn: 0.18 }},
    {{ name: "Suzlon Energy", ticker: "SUZLON.NS", price: 54, annualReturn: 0.42 }},
    {{ name: "Artemis Medicare", ticker: "ARTEMISMED.NS", price: 175, annualReturn: 0.28 }},
    {{ name: "Aditya Birla Capital", ticker: "ABCAPITAL.NS", price: 180, annualReturn: 0.14 }},
    {{ name: "Motherson Sumi", ticker: "MSUMI.NS", price: 62, annualReturn: 0.12 }},
    {{ name: "Muthoot Microfin", ticker: "MUTHOOTMF.NS", price: 230, annualReturn: 0.10 }},
    {{ name: "Tracxn Technologies", ticker: "TRACXN.NS", price: 92, annualReturn: 0.08 }},
    {{ name: "Reliance Industries", ticker: "RELIANCE.NS", price: 2950, annualReturn: 0.16 }},
    {{ name: "TCS", ticker: "TCS.NS", price: 4100, annualReturn: 0.15 }},
    {{ name: "HDFC Bank", ticker: "HDFCBANK.NS", price: 1450, annualReturn: 0.11 }},
    {{ name: "Infosys", ticker: "INFY.NS", price: 1850, annualReturn: 0.14 }},
    {{ name: "Tata Motors", ticker: "TATAMOTORS.NS", price: 980, annualReturn: 0.32 }},
    {{ name: "ICICI Bank", ticker: "ICICIBANK.NS", price: 1220, annualReturn: 0.19 }},
    {{ name: "State Bank of India", ticker: "SBIN.NS", price: 810, annualReturn: 0.22 }},
    {{ name: "Bharti Airtel", ticker: "BHARTIARTL.NS", price: 1650, annualReturn: 0.26 }},
    {{ name: "Larsen & Toubro", ticker: "LT.NS", price: 3600, annualReturn: 0.20 }},
    {{ name: "Axis Bank", ticker: "AXISBANK.NS", price: 1180, annualReturn: 0.17 }},
    {{ name: "Kotak Mahindra Bank", ticker: "KOTAKBANK.NS", price: 1780, annualReturn: 0.12 }},
    {{ name: "ITC", ticker: "ITC.NS", price: 490, annualReturn: 0.18 }},
    {{ name: "Hindustan Unilever", ticker: "HINDUNILVR.NS", price: 2750, annualReturn: 0.10 }},
    {{ name: "Bajaj Finance", ticker: "BAJFINANCE.NS", price: 7200, annualReturn: 0.22 }},
    {{ name: "Maruti Suzuki", ticker: "MARUTI.NS", price: 12400, annualReturn: 0.16 }},
    {{ name: "Asian Paints", ticker: "ASIANPAINT.NS", price: 3100, annualReturn: 0.09 }},
    {{ name: "Mahindra & Mahindra", ticker: "M&M.NS", price: 2800, annualReturn: 0.35 }},
    {{ name: "Titan Company", ticker: "TITAN.NS", price: 3500, annualReturn: 0.21 }},
    {{ name: "Sun Pharma", ticker: "SUNPHARMA.NS", price: 1700, annualReturn: 0.24 }},
    {{ name: "UltraTech Cement", ticker: "ULTRACEMCO.NS", price: 11200, annualReturn: 0.18 }},
    {{ name: "NTPC", ticker: "NTPC.NS", price: 410, annualReturn: 0.30 }},
    {{ name: "Power Grid Corp", ticker: "POWERGRID.NS", price: 340, annualReturn: 0.25 }},
    {{ name: "Tata Steel", ticker: "TATASTEEL.NS", price: 155, annualReturn: 0.18 }},
    {{ name: "Coal India", ticker: "COALINDIA.NS", price: 490, annualReturn: 0.28 }},
    {{ name: "HCL Technologies", ticker: "HCLTECH.NS", price: 1750, annualReturn: 0.19 }},
    {{ name: "Wipro", ticker: "WIPRO.NS", price: 520, annualReturn: 0.12 }},
    {{ name: "Tech Mahindra", ticker: "TECHM.NS", price: 1580, annualReturn: 0.15 }},
    {{ name: "ONGC", ticker: "ONGC.NS", price: 290, annualReturn: 0.32 }},
    {{ name: "JSW Steel", ticker: "JSWSTEEL.NS", price: 930, annualReturn: 0.16 }},
    {{ name: "Adani Enterprises", ticker: "ADANIENT.NS", price: 3100, annualReturn: 0.25 }},
    {{ name: "Adani Ports", ticker: "ADANIPORTS.NS", price: 1450, annualReturn: 0.28 }},
    {{ name: "Bharat Electronics", ticker: "BEL.NS", price: 290, annualReturn: 0.45 }},
    {{ name: "Hindustan Aeronautics", ticker: "HAL.NS", price: 4700, annualReturn: 0.50 }},
    {{ name: "Trent", ticker: "TRENT.NS", price: 7100, annualReturn: 0.60 }},
    {{ name: "Zomato", ticker: "ZOMATO.NS", price: 260, annualReturn: 0.55 }},
    {{ name: "Jio Financial Services", ticker: "JIOFIN.NS", price: 340, annualReturn: 0.20 }},
    {{ name: "REC Limited", ticker: "RECLTD.NS", price: 620, annualReturn: 0.48 }},
    {{ name: "PFC Limited", ticker: "PFC.NS", price: 540, annualReturn: 0.46 }},
    {{ name: "IRFC", ticker: "IRFC.NS", price: 175, annualReturn: 0.52 }},
    {{ name: "Indian Oil Corp", ticker: "IOC.NS", price: 170, annualReturn: 0.22 }},
    {{ name: "BPCL", ticker: "BPCL.NS", price: 350, annualReturn: 0.20 }},
    {{ name: "Varun Beverages", ticker: "VBL.NS", price: 610, annualReturn: 0.40 }},
    {{ name: "DLF Limited", ticker: "DLF.NS", price: 860, annualReturn: 0.28 }}
];

const BENCHMARKS = {{ nifty: 0.14, gold: 0.12, sp500: 0.13 }};
let lineChart = null, pieChart = null;

function addStockRow(defaultTicker = '', defaultQty = 10) {{
    const list = document.getElementById('stocksList');
    if (list.children.length >= 10) {{ alert('Maximum 10 stocks allowed'); return; }}

    const row = document.createElement('div');
    row.className = 'stock-row';

    let optionsHTML = STOCK_DB.map(s =>
        `<option value="${{s.ticker}}" ${{s.ticker === defaultTicker ? 'selected' : ''}}>${{s.name}} (${{s.ticker.replace('.NS','')}})</option>`
    ).join('');

    row.innerHTML = `
        <select class="stock-select">${{optionsHTML}}</select>
        <input type="number" class="stock-qty" value="${{defaultQty}}" min="1" placeholder="Qty">
        <button class="btn-remove">✕</button>
    `;

    row.querySelector('.btn-remove').onclick = () => row.remove();
    list.appendChild(row);
}}

function calculate() {{
    const years = Math.max(1, parseInt(document.getElementById('timeHorizonInput').value) || 1);
    const rows = document.querySelectorAll('.stock-row');
    let holdings = [], totalCurrent = 0, initialVal = 0;
    let logSummaryArray = [];

    rows.forEach(r => {{
        const ticker = r.querySelector('.stock-select').value;
        const q = parseFloat(r.querySelector('.stock-qty').value) || 0;
        const stockInfo = STOCK_DB.find(s => s.ticker === ticker);

        if (stockInfo && q > 0) {{
            const currentVal = stockInfo.price * q;
            const startPrice = stockInfo.price / Math.pow(1 + stockInfo.annualReturn, years);
            initialVal += startPrice * q;
            totalCurrent += currentVal;

            holdings.push({{
                ...stockInfo,
                qty: q,
                startPrice: startPrice,
                currentVal: currentVal
            }});
            logSummaryArray.push(`${{stockInfo.name}} (${{q}} qty)`);
        }}
    }});

    if (!holdings.length) return alert('Select at least one stock');

    const months = years * 12;
    const labels = [], portNorm = [], niftyNorm = [], goldNorm = [], spNorm = [];
    let best = holdings[0], maxRet = -999;

    holdings.forEach(h => {{
        const ret = (Math.pow(1 + h.annualReturn, years) - 1) * 100;
        if (ret > maxRet) {{ maxRet = ret; best = h; }}
    }});

    for (let m = 0; m <= months; m++) {{
        labels.push(`M${{m}}`);
        let mVal = 0;
        holdings.forEach(h => {{
            const mRate = h.annualReturn / 12;
            const priceAtM = (h.price / Math.pow(1 + h.annualReturn, years)) * Math.pow(1 + mRate, m);
            mVal += priceAtM * h.qty;
        }});
        portNorm.push((mVal / initialVal) * 100);
        niftyNorm.push(Math.pow(1 + BENCHMARKS.nifty / 12, m) * 100);
        goldNorm.push(Math.pow(1 + BENCHMARKS.gold / 12, m) * 100);
        spNorm.push(Math.pow(1 + BENCHMARKS.sp500 / 12, m) * 100);
    }}

    const totalRet = ((totalCurrent - initialVal) / initialVal) * 100;
    const cagr = (Math.pow(totalCurrent / initialVal, 1 / years) - 1) * 100;

    document.getElementById('valCurrent').innerText = `₹${{Math.round(totalCurrent).toLocaleString('en-IN')}}`;
    document.getElementById('valReturn').innerText = `${{totalRet >= 0 ? '+' : ''}}${{totalRet.toFixed(1)}}%`;
    document.getElementById('valReturn').style.color = totalRet >= 0 ? '#10B981' : '#EF4444';
    document.getElementById('valCAGR').innerText = `${{cagr.toFixed(1)}}%`;
    document.getElementById('valBest').innerText = `${{best.name}} (+${{maxRet.toFixed(0)}}%)`;

    renderTable(holdings, totalCurrent);
    renderCharts(labels, portNorm, niftyNorm, goldNorm, spNorm, holdings, totalCurrent);

    // Send payload to Google Sheets Web App
    if (LOGGING_WEBAPP_URL && LOGGING_WEBAPP_URL !== "YOUR_GOOGLE_APPS_SCRIPT_WEB_APP_URL_HERE") {{
        fetch(LOGGING_WEBAPP_URL, {{
            method: 'POST',
            mode: 'no-cors',
            headers: {{ 'Content-Type': 'application/json' }},
            body: JSON.stringify({{
                years: years,
                stocks: logSummaryArray.join(", "),
                totalValue: `₹${{Math.round(totalCurrent).toLocaleString('en-IN')}}`,
                totalReturn: `${{totalRet.toFixed(1)}}%`,
                cagr: `${{cagr.toFixed(1)}}%`
            }})
        }}).catch(err => console.log("Analytics logging error:", err));
    }}
}}

function renderTable(holdings, totalCurrent) {{
    const tbody = document.getElementById('holdingsTableBody');
    tbody.innerHTML = '';

    holdings.forEach(h => {{
        const weightPct = ((h.currentVal / totalCurrent) * 100).toFixed(1);
        const tr = document.createElement('tr');
        tr.innerHTML = `
            <td style="font-weight:700; color:var(--im-gold);">${{h.name}}</td>
            <td>${{h.qty}}</td>
            <td>₹${{Math.round(h.startPrice)}}</td>
            <td style="color:#10B981; font-weight:700;">₹${{h.price}}</td>
            <td>₹${{Math.round(h.currentVal).toLocaleString('en-IN')}}</td>
            <td style="color:var(--im-yellow); font-weight:700;">${{weightPct}}%</td>
        `;
        tbody.appendChild(tr);
    }});
}}

function renderCharts(labels, port, nifty, gold, sp, holdings, totalVal) {{
    const ctxL = document.getElementById('lineChart').getContext('2d');
    if (lineChart) lineChart.destroy();
    lineChart = new Chart(ctxL, {{
        type: 'line',
        data: {{
            labels,
            datasets: [
                {{ label: 'Portfolio', data: port, borderColor: '#FFD700', backgroundColor: 'rgba(255, 215, 0, 0.1)', borderWidth: 3, pointRadius: 0, fill: true }},
                {{ label: 'Nifty 50', data: nifty, borderColor: '#C8102E', borderWidth: 2, pointRadius: 0 }},
                {{ label: 'Gold', data: gold, borderColor: '#FFA700', borderWidth: 2, pointRadius: 0 }},
                {{ label: 'S&P 500', data: sp, borderColor: '#FFFFFF', borderWidth: 1.5, borderDash: [4, 4], pointRadius: 0 }}
            ]
        }},
        options: {{
            responsive: true,
            maintainAspectRatio: false,
            scales: {{
                x: {{ grid: {{ color: '#2A2A32' }}, ticks: {{ color: '#AAAAAA' }} }},
                y: {{ grid: {{ color: '#2A2A32' }}, ticks: {{ color: '#AAAAAA' }} }}
            }},
            plugins: {{ legend: {{ labels: {{ color: '#FFFFFF' }} }} }}
        }}
    }});

    const ctxP = document.getElementById('pieChart').getContext('2d');
    if (pieChart) pieChart.destroy();
    pieChart = new Chart(ctxP, {{
        type: 'pie',
        data: {{
            labels: holdings.map(h => h.name),
            datasets: [{{
                data: holdings.map(h => ((h.currentVal / totalVal) * 100).toFixed(1)),
                backgroundColor: ['#C8102E', '#FFD700', '#FFA700', '#7A0010', '#D4AF37', '#8B0000', '#E6C200', '#B22222', '#F1C40F', '#900C3F']
            }}]
        }},
        options: {{
            responsive: true,
            maintainAspectRatio: false,
            plugins: {{
                legend: {{ position: 'bottom', labels: {{ color: '#FFFFFF', boxWidth: 10, font: {{ size: 10 }} }} }}
            }}
        }}
    }});
}}

document.getElementById('btnAddStock').onclick = () => addStockRow();
document.getElementById('btnCalculate').onclick = calculate;

// Initial setup on load
addStockRow("IDFCFIRSTB.NS", 165);
addStockRow("SUZLON.NS", 145);
addStockRow("ARTEMISMED.NS", 55);
calculate();
</script>
</body>
</html>
"""

HTML(colab_nse_app_code)

Stock,Qty,Start Price,Traded Price,Value,Weight
